In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from sppaper.common.plot import setup_matplotlib_params, find_font_path

setup_matplotlib_params()

import pickle
from pathlib import Path


import numpy as np
import cv2
import h5py
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cmasher
import imageio.v3 as iio
from PIL import Image, ImageDraw, ImageFont
from scipy.signal import medfilt
from scipy.signal import savgol_filter
from vidstab import VidStab
from tqdm import trange, tqdm

from flygym.anatomy import LEGS

import sppaper.kinematics.shared_constants as const
from sppaper.kinematics.data import align_smooth_decompose_trajs
from sppaper.common.io import load_precise_sparse_frames
from sppaper.common.resources import get_outputs_dir

KIN_FILTER_WINDOW_SIZE = 3
DOF_DISPLAY_NAMES = {
    "ThC_pitch": "ThC-p",
    "ThC_roll": "ThC-r",
    "ThC_yaw": "ThC-y",
    "CTr_pitch": "CTr-p",
    "CTr_roll": "CTr-r",
    "FTi_pitch": "FTi-p",
    "TiTa_pitch": "TiTa-p",
}
LEG_DISP_NAMES = {
    "LF": "Left front leg",
    "LM": "Left middle leg",
    "LH": "Left hind leg",
    "RF": "Right front leg",
    "RM": "Right middle leg",
    "RH": "Right hind leg",
}
AXIS_DISPLAY_NAMES = {"x": "fore/aft", "y": "med/lat", "z": "height"}
MM_TO_IN = 1 / 25.4

CLAW_XYZ_COLOR = "#546a76"
DOF_ANGLES_COLOR = "#a23e48"
ACTUATOR_FORCES_COLOR = "#689829"
REC_COLOR = "#546a76"
SIM_COLOR = "#689829"

VIZ_OUTPUT_DIR = get_outputs_dir() / "neuromechfly_replay/visualization/"



POSE_WORKING_DIM = 256
RAW_INPUT_CROP_DIM = 900
SWING_SPEED_THRESHOLD = 25  # mm/s, threshold for when the leg in swing vs stance

In [ ]:
def undo_transform(xy_posttransform: np.ndarray, transform_matrices: np.ndarray) -> np.ndarray:
    """
    Reverses affine transforms applied to 2D point coordinates.

    Args:
        xy_posttransform:   (n_frames, ..., 2) — transformed coordinates
        transform_matrices: (n_frames, 2, 3)   — affine matrices that were applied

    Returns:
        xy_pretransform: (n_frames, ..., 2) — recovered original coordinates
    """
    n_frames = xy_posttransform.shape[0]
    original_shape = xy_posttransform.shape

    # Flatten all middle dims into a single n_points dimension
    xy_flat = xy_posttransform.reshape(n_frames, -1, 2)  # (n_frames, n_points, 2)

    # Promote each 2x3 matrix to a full 3x3 affine matrix by appending [0, 0, 1]
    bottom_row = np.tile(np.array([[[0.0, 0.0, 1.0]]]), (n_frames, 1, 1))  # (n_frames, 1, 3)
    M_3x3 = np.concatenate([transform_matrices, bottom_row], axis=1)       # (n_frames, 3, 3)

    # Invert all matrices at once
    M_inv = np.linalg.inv(M_3x3)  # (n_frames, 3, 3)

    # Convert points to homogeneous coordinates: (n_frames, n_points, 3)
    ones = np.ones((*xy_flat.shape[:2], 1))
    xy_homogeneous = np.concatenate([xy_flat, ones], axis=-1)

    # Apply inverse transform: einsum over (frame, point, coord)
    xy_pretransform = np.einsum("fij,fpj->fpi", M_inv, xy_homogeneous)

    # Drop the homogeneous coordinate and restore the original shape
    return xy_pretransform[..., :2].reshape(original_shape)

In [ ]:
def get_coords_arena_mm(kinematic_snippet):
    slice_ = slice(kinematic_snippet.start_idx, kinematic_snippet.end_idx)

    poseforge_output_dir = kinematic_snippet.exp_trial_dir / "poseforge_output"
    with h5py.File(poseforge_output_dir / "inverse_kinematics_output.h5", "r") as f:
        raw_world_xyz = f["rawpred_world_xyz"][slice_, :30, :].reshape(-1, 6, 5, 3)
        fwdkin_world_xyz = f["fwdkin_world_xyz"][slice_, :30, :].reshape(-1, 6, 5, 3)
    with h5py.File(poseforge_output_dir / "keypoints3d_prediction.h5", "r") as f:
        cam_xy_px = f["pred_xy"][slice_, :30, :].reshape(-1, 6, 5, 2)
    assert raw_world_xyz.shape[0] == len(kinematic_snippet)
    assert cam_xy_px.shape[0] == len(kinematic_snippet)
    assert fwdkin_world_xyz.shape[0] == len(kinematic_snippet)

    # Image was downsampled before given as input to neural network. Scale output px
    # coords back up to original input image scale
    cam_xy_px *= RAW_INPUT_CROP_DIM / POSE_WORKING_DIM
    # ... then undo alignment and cropping
    cam_xy_px_pretransform = undo_transform(cam_xy_px, kinematic_snippet.crop_transmats)

    mapper = kinematic_snippet.spotlight_coords_mapper
    stage_pos_mm_expanded = np.tile(
        kinematic_snippet.stage_pos_mm[:, None, None, :], (1, 6, 5, 1)
    )
    xypos_arena = mapper.stage_and_pixel_to_physical(
        stage_pos_mm_expanded, cam_xy_px_pretransform
    )

    return xypos_arena


In [ ]:
def get_swing_mask(
    sim_dir: Path,
    t_range: tuple[float, float] | None = None,
    speed_threshold=SWING_SPEED_THRESHOLD,
    debug_plot_path=None,
):
    with open(sim_dir / "sim_data.pkl", "rb") as f:
        data = pickle.load(f)
        kinematic_snippet = data["snippet"]
    if t_range is not None:
        kinematic_snippet = kinematic_snippet.get_subselection(*t_range)

    kpts_xypos_arena = get_coords_arena_mm(kinematic_snippet)
    # kpts_xypos_arena: (n_frames, 6 legs, 5 kpts per leg, {x,y})
    claw_xypos_arena = kpts_xypos_arena[:, :, -1, :]

    dt = 1 / kinematic_snippet.data_fps
    claw_xyvel_arena = np.gradient(claw_xypos_arena, dt, axis=0)
    claw_speed_arena = np.linalg.norm(claw_xyvel_arena, axis=-1)
    moving_mask = claw_speed_arena > speed_threshold
    claw_xypos_arena_movingonly = claw_xypos_arena.copy()
    claw_xypos_arena_movingonly[~moving_mask, :] = np.nan
    claw_xypos_arena_stationaryonly = claw_xypos_arena.copy()
    claw_xypos_arena_stationaryonly[moving_mask, :] = np.nan

    if debug_plot_path is not None:
        debug_plot_path = Path(debug_plot_path)
        debug_plot_path.parent.mkdir(parents=True, exist_ok=True)
        fig, ax = plt.subplots(figsize=(3, 3), tight_layout=True)
        for i in range(6):
            ax.plot(
                claw_xypos_arena_movingonly[:, i, 0],
                claw_xypos_arena_movingonly[:, i, 1],
                linewidth=1,
                color="#546a76",
                label="Swing" if i == 0 else None,
            )
            ax.plot(
                claw_xypos_arena_stationaryonly[:, i, 0],
                claw_xypos_arena_stationaryonly[:, i, 1],
                linewidth=1,
                color="#a23e48",
                zorder=10,
                label="Stance" if i == 0 else None,
            )
        ax.legend(bbox_to_anchor=(1.04, 1), loc="upper left")
        ax.set_aspect("equal")
        ax.set_title("Claw trajectories in arena")
        ax.set_xlabel("x (mm)")
        ax.set_ylabel("y (mm)")
        fig.savefig(debug_plot_path)
        plt.close(fig)

    return moving_mask

In [ ]:
from sppaper.common.resources import get_outputs_dir

VISUALIZED_SIM_DIR = (
    get_outputs_dir() / "neuromechfly_replay/kp150_damp0.5_slidfric2.0/snippet21/"
)
VISUALIZED_SIM_TIMERANGE = (0.5, 2.5)
VIZ_OUTPUT_DIR = get_outputs_dir() / "neuromechfly_replay/visualization/"
VIZ_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
from sppaper.common.filter import median_filter_over_time
from flygym.anatomy import LEGS

swing_mask = get_swing_mask(
    VISUALIZED_SIM_DIR,
    t_range=VISUALIZED_SIM_TIMERANGE,
    debug_plot_path=VIZ_OUTPUT_DIR / "gait_diagram_debug.pdf",
    speed_threshold=12,
)
swing_mask_filtered = median_filter_over_time(swing_mask.astype(np.float32), 11).astype(
    np.bool_
)
fig, ax = plt.subplots(figsize=(3, 1.1), tight_layout=True)
ax.imshow(
    ~swing_mask_filtered.T,
    aspect="auto",
    interpolation="none",
    extent=[0, VISUALIZED_SIM_TIMERANGE[1] - VISUALIZED_SIM_TIMERANGE[0], 6, 0],
    cmap="gray",
)
ax.set_xlabel("Time (s)")
ax.set_yticks(np.arange(6) + 0.5, [x.upper() for x in LEGS])
ax.set_title("Gait diagram (black = stance, white = swing)")

In [ ]:
swing_mask